In [1]:
import os
import sys 
from pathlib import Path
from dotenv import load_dotenv
import numpy as np
load_dotenv(override=True)
ROOT = Path.cwd().parent.parent.parent.resolve()

print(f"ROOT: {ROOT}")
sys.path.append(str(ROOT))

ROOT: /users/oshan/Dev/financial-document-based-agent-system


In [2]:
from dochandler.main import ExTrRAGDocHandler
from cgcore.vectordb.milvus import MilvusDB
from cgcore.embedder.openai import OpenAIEmbedder
from cgcore.llm.openai import OpenAILlm

from cgcore.configs.vectordb.milvus import MilvusConfig
from cgcore.configs.embedder.openai import OpenAIEmbedderConfig
from cgcore.configs.llm.openai import OpenAILlmConfig

In [3]:
llm_config = OpenAILlmConfig(api_key=os.getenv('OPENAI_API_KEY'))
embedder_config = OpenAIEmbedderConfig(api_key=os.getenv('OPENAI_API_KEY'), model='text-davinci-003', dimesion=os.getenv('MONGO_DB_DIMENSION'))
vectordb_config = MilvusConfig(
                collection_name=os.getenv('MILVUS_COLLECTION_NAME'),
                dimensions=1536,  # Set explicit dimension value
                )


In [4]:
openai = OpenAILlm(llm_config)
embeder = OpenAIEmbedder(embedder_config)
vectordb = MilvusDB(vectordb_config)

AsyncMilvusClient initialized.


/tmp/ipykernel_3901671/1873515904.py:1: UserWarning: Parameters {'top_p'} should be specified explicitly. Instead they were passed in as part of `model_kwargs` parameter.
  openai = OpenAILlm(llm_config)
/users/oshan/Dev/financial-document-based-agent-system/.venv/lib/python3.12/site-packages/langchain_openai/embeddings/base.py:313: UserWarning: WARNING! encoding_format is not default parameter.
                    encoding_format was transferred to model_kwargs.
                    Please confirm that encoding_format is what you intended.
  warnings.warn(


Collection 'financial_documents_backend_test' already exists. Skipping creation.


## ExTr RAG 

In [7]:
extr_rag = ExTrRAGDocHandler(
    llm=openai,
    embedder=embeder,
    db=vectordb,
    memory="none",
    history=True
)

## Document Loading test


In [8]:
extr_rag.loader.dry_run = False

In [9]:
pdf_path = "/users/oshan/Dev/financial-document-based-agent-system/FIU_AR_2011.pdf"

In [10]:
print(f"Testing Processing for: {os.path.basename(pdf_path)} ---")

Testing Processing for: FIU_AR_2011.pdf ---


In [10]:
# from dochandler.src.loader.dockling_loader import DocklingLoader
# from dochandler.src.chunkers.mdx import MdxChunker

# loader = DocklingLoader(server_url="http://localhost:8080/documents/convert")
# # creates the .md file locally
# raw_data = loader.load_data(pdf_path) 
# print(f"Conversion complete. Markdown size: {len(raw_data['data'][0]['content'])} chars")

# chunker = MdxChunker() 
# chunks = chunker.create_chunks(loader, pdf_path)

# print(f"\n HUNKING RESULTS:")
# print(f"Total Chunks Created: {len(chunks['ids'])}")


In [11]:
# for i in range(min(30, len(chunks['ids']))):
#     chunk_id = chunks['ids'][i]
#     content = chunks['documents'][i]
#     meta = chunks['metadatas'][i]
    
#     print(f"\n CHUNK #{i} [ID: {chunk_id}]")
#     print(f"   Metadata: {meta}")
#     print(f"   {content.strip()[:300]}...") 



### Questions generation and saving in the VDB

In [12]:
records = await extr_rag.add_document(pdf_path)

Loading 'FIU_AR_2011.pdf' via Dockling Server...


/users/oshan/Dev/financial-document-based-agent-system/.venv/lib/python3.12/site-packages/langchain_openai/chat_models/base.py:2067: UserWarning: Cannot use method='json_schema' with model gpt-3.5-turbo since it doesn't support OpenAI's Structured Output API. You can see supported models here: https://platform.openai.com/docs/guides/structured-outputs#supported-models. To fix this warning, set `method='function_calling'. Overriding to method='function_calling'.
  warnings.warn(
ic| questions: ['What is the main highlight of the Annual Report 2011?',
                'Can you provide a summary of the financial performance in the Annual Report '
                '2011?',
                'What are the key achievements mentioned in the Annual Report 2011?']
ic| questions: ['What is the Financial Intelligence Unit of Sri Lanka responsible for?',
                'When was the Annual Report of the Financial Intelligence Unit of Sri Lanka '
                'published?',
                'Which or

Successfully inserted 91 records into Milvus.
